# Pipeline Serie A - Limpieza automatizada
*Parámetros*: equipo="ALL", jornada=None, fecha="2026-03-21"

## PIPELINE LIMPIEZA SERIE A TIM 2025/26
**Notebook parametrizado para Papermill**  
**Input**: data/serie_a_2526/*.csv (de scraping)  
**Output**: data/cleaned/*.csv + validaciones


## Imports + Config

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración paths
BASE_PATH = Path("data")
RAW_PATH = BASE_PATH / "serie_a_2526"  # Cambiado para usar los datos scrapeados
CLEAN_PATH = BASE_PATH / "cleaned"
ASSETS_PATH = Path("assets") / "escudos"

# Parámetros (para Papermill)
equipo = "ALL"  # "ALL" o nombre de equipo
jornada = None  # Número de ronda o None para todas
fecha = "2026-03-21"  # Fecha límite o None para todas
season_filter = "25/26"  # Temporada a filtrar (ej. "25/26")

print(f"Parámetros: equipo={equipo}, jornada={jornada}, fecha={fecha}, season={season_filter}")

# Config Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Paths configurados:")
print(f"Raw: {RAW_PATH}")
print(f"Clean: {CLEAN_PATH}")
print(f"Escudos: {ASSETS_PATH}")


Parámetros: equipo=ALL, jornada=None, fecha=2026-03-21
✅ Paths configurados:
Raw: data\serie_a_2526
Clean: data\cleaned
Escudos: assets\escudos


## Carga CSV  → shapes iniciales

### CARGA DATOS RAW (últimos archivos)

In [ ]:
# Buscar archivos más recientes
csv_files = list(RAW_PATH.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No hay CSV en data/raw/")

print("Archivos encontrados:")
for f in csv_files:
    print(f"  - {f.name} ({f.stat().st_size/1e3:.0f} KB)")

# Cargar por nombre (ajusta según tus archivos)
matches_info = pd.read_csv(RAW_PATH / "MatchesInformation.csv")
player_stats = pd.read_csv(RAW_PATH / "PlayerStatistics.csv")
shot_map = pd.read_csv(RAW_PATH / "ShotMap.csv")
team_stats = pd.read_csv(RAW_PATH / "TeamStatistics.csv")

# Aplicar filtros de parámetros
if equipo != "ALL":
    matches_info = matches_info[(matches_info['home_team'] == equipo) | (matches_info['away_team'] == equipo)]
    print(f"Filtrado por equipo: {equipo}")

if jornada is not None:
    matches_info = matches_info[matches_info['round'] == jornada]
    print(f"Filtrado por jornada: {jornada}")

if season_filter is not None:
    matches_info = matches_info[matches_info['season'] == season_filter]
    print(f"Filtrado por temporada: {season_filter}")

print("\nShapes iniciales:")
print(f"Matches: {matches_info.shape}")
print(f"Players: {player_stats.shape}")
print(f"Shots: {shot_map.shape}")
print(f"Teams: {team_stats.shape}")

Archivos encontrados:
  - MatchesInformation.csv (17 KB)
  - PlayerStatistics.csv (2865 KB)
  - ShotMap.csv (634 KB)
  - TeamStatistics.csv (134 KB)

Shapes iniciales:
Matches: (305, 9)
Players: (14029, 81)
Shots: (7458, 15)
Teams: (610, 50)


## Funciones

### PIPELINE LIMPIEZA MATCHES


### Función clean_matches()

In [3]:
def clean_matches(df):
    """Limpia scores nulos y valida tipos"""
    print(f"📊 Antes: {len(df)} filas")
    
    df_clean = (df.dropna(subset=["home_score", "away_score"])
                   .assign(home_score=lambda x: x["home_score"].astype(int),
                          away_score=lambda x: x["away_score"].astype(int))
                   .reset_index(drop=True))
    
    print(f"✅ Después: {len(df_clean)} filas válidas")
    return df_clean

matches_clean = clean_matches(matches_info)
matches_clean.head(3)


📊 Antes: 305 filas
✅ Después: 300 filas válidas


,id,tournament,season,round,status,home_team,away_team,home_score,away_score
0,13981424,Serie A,25/26,1,Ended,Juventus,Parma,2,0
1,13981438,Serie A,25/26,1,Ended,Udinese,Hellas Verona,1,1
2,13981421,Serie A,25/26,1,Ended,Inter,Torino,5,0


### FILTRADO CONSISTENTE (solo matches válidos)

In [4]:
valid_ids = matches_clean['id'].unique()
print(f"🎯 {len(valid_ids)} partidos válidos")

# Filtrar TODOS los datasets
player_stats = player_stats[player_stats['match'].isin(valid_ids)]
shot_map = shot_map[shot_map['match'].isin(valid_ids)]
team_stats = team_stats[team_stats['match'].isin(valid_ids)]

print("✅ Datasets filtrados:")
print(f"Players: {player_stats.shape}")
print(f"Shots: {shot_map.shape}")
print(f"Teams: {team_stats.shape}")


🎯 300 partidos válidos
✅ Datasets filtrados:
Players: (14029, 81)
Shots: (7458, 15)
Teams: (600, 50)


### LIMPIEZA PLAYER & TEAM STATS

In [5]:
def clean_numeric_stats(df, cols=None):
    """Rellena NaN con 0 y pasa a int"""
    if cols is None:
        cols = df.select_dtypes(include=['float64']).columns
        
    df_clean = df.copy()
    df_clean[cols] = df_clean[cols].fillna(0).astype(int)
    return df_clean

# Aplicar
player_stats_clean = clean_numeric_stats(player_stats)
team_stats_clean = clean_numeric_stats(team_stats)

print("✅ Stats numéricas limpias")


✅ Stats numéricas limpias


## Validaciones (posesión=100%)


In [6]:
# Posesión debe sumar 100%
possession_check = team_stats_clean.groupby('match')['ballPossession'].sum()
invalid = (possession_check != 100).sum()
print(f"⚽ Posesión OK: {100-invalid}/{len(possession_check)} partidos")

# Guardar cleaned
CLEAN_PATH.mkdir(exist_ok=True)
matches_clean.to_csv(CLEAN_PATH / "matches_clean.csv", index=False)
player_stats_clean.to_csv(CLEAN_PATH / "player_stats_clean.csv", index=False)
shot_map.to_csv(CLEAN_PATH / "shot_map_clean.csv", index=False)
team_stats_clean.to_csv(CLEAN_PATH / "team_stats_clean.csv", index=False)

print("💾 Datasets guardados en data/cleaned/")
print("\n🎉 PIPELINE COMPLETADO ✅")


⚽ Posesión OK: 100/300 partidos
💾 Datasets guardados en data/cleaned/

🎉 PIPELINE COMPLETADO ✅
